# 11 Graphical Showcase - Python

## Biochemistry question

What kinds of scientific visuals can BioChem Data Lab create from small synthetic life-science datasets?

This notebook is a visual tour. Each figure uses synthetic data only and is meant for learning, not clinical, diagnostic, regulatory, or drug-efficacy claims.


In [1]:
import plotly.io as pio
pio.renderers.default = "iframe"


## Setup

This notebook collects a few high-interest plots from across the project. It also exports each Plotly figure to `example_outputs/*.html`.


In [2]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))
OUTPUT_DIR = PROJECT_ROOT / "example_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

from src.biochem_stats import calculate_fold_change, summary_mean_sd_sem
from src.data_quality import flag_outliers_zscore
from src.enzyme_kinetics import fit_michaelis_menten, make_prediction_table, summarize_velocity
from src.growth_curve import summarize_growth
from src.plate_qc import add_plate_position_flags, make_plate_matrix
from src.qpcr import calculate_delta_delta_ct
from src.sequence_basics import summarize_sequences
from src.standard_curve import fit_linear_standard_curve, summarize_standard_curve


def show_and_export(fig, filename):
    fig.write_html(OUTPUT_DIR / filename)
    # If a chart does not render in Jupyter, try: fig.show(renderer="browser")
    fig.show(renderer="iframe")
    return OUTPUT_DIR / filename


## 1. Enzyme Activity: Group Means with Error Bars

Context: compare synthetic enzyme activity across control and inhibitor groups.

Interpretation question: which group has the lowest mean enzyme activity?

Limitation: this plot does not prove a mechanism.


In [3]:
enzyme_df = pd.read_csv(PROJECT_ROOT / "data" / "enzyme_activity" / "enzyme_activity_three_groups.csv")
enzyme_summary = summary_mean_sd_sem(enzyme_df, "group", "enzyme_activity")
fig = px.bar(
    enzyme_summary,
    x="group",
    y="mean_value",
    error_y="sem_value",
    title="Synthetic Enzyme Activity by Group",
    labels={"mean_value": "Mean enzyme activity"},
)
show_and_export(fig, "showcase_01_enzyme_activity.html")


PosixPath('/home/scott/workspace/biochem-data-lab/example_outputs/showcase_01_enzyme_activity.html')

## 2. Dose Response: Concentration-Response Curve

Context: visualize synthetic cell viability across compound concentrations.

Interpretation question: which compound shows the larger decrease at high concentration?

Limitation: this is not drug-efficacy evidence.


In [4]:
dose_df = pd.read_csv(PROJECT_ROOT / "data" / "drug_response" / "drug_response_clear.csv")
dose_summary = (
    dose_df.groupby(["drug_name", "concentration_uM"])
    .agg(mean_viability=("cell_viability_percent", "mean"), sd_viability=("cell_viability_percent", "std"), n=("cell_viability_percent", "count"))
    .reset_index()
)
dose_summary["sem_viability"] = dose_summary["sd_viability"] / (dose_summary["n"] ** 0.5)
dose_summary["plot_concentration"] = dose_summary["concentration_uM"].replace(0, 0.001)
fig = px.line(
    dose_summary,
    x="plot_concentration",
    y="mean_viability",
    color="drug_name",
    markers=True,
    error_y="sem_viability",
    title="Synthetic Dose-Response Curves",
)
fig.update_xaxes(type="log", title="Concentration (uM, log scale; 0 plotted as 0.001)")
fig.update_yaxes(title="Mean cell viability (%)")
show_and_export(fig, "showcase_02_dose_response.html")


PosixPath('/home/scott/workspace/biochem-data-lab/example_outputs/showcase_02_dose_response.html')

## 3. QC Scatter: Suspicious Replicates

Context: show individual assay points and flag values for review.

Interpretation question: which points should be reviewed before summarizing?

Limitation: flags are review prompts, not automatic deletion decisions.


In [5]:
qc_df = pd.read_csv(PROJECT_ROOT / "data" / "drug_response" / "drug_response_outlier.csv")
qc_flagged = flag_outliers_zscore(qc_df, ["drug_name", "concentration_uM"], "cell_viability_percent", threshold=1.0)
qc_flagged["plot_concentration"] = qc_flagged["concentration_uM"].replace(0, 0.001)
fig = px.scatter(
    qc_flagged,
    x="plot_concentration",
    y="cell_viability_percent",
    color="qc_flag",
    symbol="drug_name",
    hover_data=["sample_id", "replicate", "z_score"],
    title="Synthetic QC Scatter Plot",
)
fig.update_xaxes(type="log", title="Concentration (uM, log scale; 0 plotted as 0.001)")
show_and_export(fig, "showcase_03_qc_scatter.html")


PosixPath('/home/scott/workspace/biochem-data-lab/example_outputs/showcase_03_qc_scatter.html')

## 4. Gene Expression Heatmap

Context: visualize toy gene expression patterns across control, low-treatment, and high-treatment samples.

Interpretation question: do sample groups show visible pattern differences?

Limitation: this is not a full RNA-seq workflow.


In [6]:
expr_df = pd.read_csv(PROJECT_ROOT / "data" / "gene_expression" / "gene_expression_matrix_pca.csv")
expr = expr_df.set_index("gene")
expr_z = expr.sub(expr.mean(axis=1), axis=0).div(expr.std(axis=1), axis=0)
fig = px.imshow(
    expr_z,
    aspect="auto",
    title="Toy Gene Expression Heatmap",
    labels={"x": "Sample", "y": "Gene", "color": "z-score"},
)
show_and_export(fig, "showcase_04_gene_heatmap.html")


PosixPath('/home/scott/workspace/biochem-data-lab/example_outputs/showcase_04_gene_heatmap.html')

## 5. Plate QC Heatmap

Context: inspect a synthetic 96-well-style plate for spatial patterns.

Interpretation question: do edge wells look different from interior wells?

Limitation: real plate QC needs protocol-specific acceptance criteria.


In [7]:
plate_df = pd.read_csv(PROJECT_ROOT / "data" / "assay_qc" / "plate_layout_edge_effect.csv")
plate_qc = add_plate_position_flags(plate_df)
plate_matrix = make_plate_matrix(plate_qc)
fig = px.imshow(
    plate_matrix,
    aspect="auto",
    title="Synthetic 96-well Plate QC Heatmap",
    labels={"x": "Column", "y": "Row", "color": "Cell viability (%)"},
)
show_and_export(fig, "showcase_05_plate_qc_heatmap.html")


PosixPath('/home/scott/workspace/biochem-data-lab/example_outputs/showcase_05_plate_qc_heatmap.html')

## 6. Michaelis-Menten-style Enzyme Kinetics

Context: show saturation behavior as substrate concentration increases.

Interpretation question: where does velocity begin to level off?

Limitation: fitted parameters are educational estimates from synthetic data.


In [8]:
kin_df = pd.read_csv(PROJECT_ROOT / "data" / "enzyme_kinetics" / "michaelis_menten_clean.csv")
kin_summary = summarize_velocity(kin_df)
kin_params = fit_michaelis_menten(kin_summary)
kin_curve = make_prediction_table(kin_params, kin_summary["substrate_mM"].min(), kin_summary["substrate_mM"].max())
fig = px.scatter(
    kin_summary,
    x="substrate_mM",
    y="mean_velocity",
    error_y="sem_velocity",
    title="Synthetic Michaelis-Menten-style Curve",
    labels={"substrate_mM": "Substrate (mM)", "mean_velocity": "Mean initial velocity"},
)
fig.add_trace(go.Scatter(x=kin_curve["substrate_mM"], y=kin_curve["predicted_velocity"], mode="lines", name="Fitted curve"))
show_and_export(fig, "showcase_06_michaelis_menten.html")


PosixPath('/home/scott/workspace/biochem-data-lab/example_outputs/showcase_06_michaelis_menten.html')

## 7. Bradford-style Standard Curve

Context: estimate synthetic unknown concentrations from a calibration curve.

Interpretation question: which estimates may need review for extrapolation?

Limitation: unknown estimates are practice calculations only.


In [9]:
std_df = pd.read_csv(PROJECT_ROOT / "data" / "standard_curves" / "bradford_standard_curve.csv")
std_summary = summarize_standard_curve(std_df)
std_fit = fit_linear_standard_curve(std_df)
fig = px.scatter(
    std_summary,
    x="known_concentration_mg_ml",
    y="mean_absorbance",
    error_y="sem_absorbance",
    title="Synthetic Bradford-style Standard Curve",
    labels={"known_concentration_mg_ml": "Known concentration (mg/mL)", "mean_absorbance": "Mean absorbance at 595 nm"},
)
fig.add_scatter(x=std_summary["known_concentration_mg_ml"], y=std_fit["slope"] * std_summary["known_concentration_mg_ml"] + std_fit["intercept"], mode="lines", name="Linear fit")
show_and_export(fig, "showcase_07_standard_curve.html")


PosixPath('/home/scott/workspace/biochem-data-lab/example_outputs/showcase_07_standard_curve.html')

## 8. qPCR Relative Expression

Context: normalize synthetic Ct values to a housekeeping gene and compare relative expression.

Interpretation question: which gene is higher in the treatment condition?

Limitation: real qPCR requires primer efficiency and control checks.


In [10]:
qpcr_df = pd.read_csv(PROJECT_ROOT / "data" / "qpcr" / "qpcr_delta_ct_sample.csv")
qpcr_results = calculate_delta_delta_ct(qpcr_df)
qpcr_plot = qpcr_results[qpcr_results["condition"] == "treatment"]
fig = px.bar(
    qpcr_plot,
    x="gene",
    y="relative_expression",
    title="Synthetic qPCR Relative Expression",
    labels={"relative_expression": "Relative expression (2^-ddCt)"},
)
show_and_export(fig, "showcase_08_qpcr_relative_expression.html")


PosixPath('/home/scott/workspace/biochem-data-lab/example_outputs/showcase_08_qpcr_relative_expression.html')

## 9. Growth Curve

Context: compare synthetic OD600 growth over time for two strains.

Interpretation question: which strain appears to increase faster during the middle of the curve?

Limitation: real growth curves need blank correction and culture details.


In [11]:
growth_df = pd.read_csv(PROJECT_ROOT / "data" / "growth_curve" / "bacterial_growth_curve.csv")
growth_summary = summarize_growth(growth_df)
fig = px.line(
    growth_summary,
    x="time_hr",
    y="mean_od600",
    color="strain",
    markers=True,
    error_y="sem_od600",
    title="Synthetic OD600 Growth Curves",
)
fig.update_yaxes(title="Mean OD600")
show_and_export(fig, "showcase_09_growth_curve.html")


PosixPath('/home/scott/workspace/biochem-data-lab/example_outputs/showcase_09_growth_curve.html')

## 10. Sequence GC Content

Context: summarize short synthetic DNA sequences.

Interpretation question: which sequence is most GC-rich?

Limitation: short sequence summaries do not prove biological function.


In [12]:
seq_df = pd.read_csv(PROJECT_ROOT / "data" / "sequences" / "synthetic_sequences.csv")
seq_summary = summarize_sequences(seq_df)
fig = px.bar(
    seq_summary,
    x="sequence_id",
    y="gc_content_percent",
    title="Synthetic Sequence GC Content",
    labels={"gc_content_percent": "GC content (%)"},
)
show_and_export(fig, "showcase_10_gc_content.html")


PosixPath('/home/scott/workspace/biochem-data-lab/example_outputs/showcase_10_gc_content.html')

## Showcase Wrap-up

What you should notice:

- Different Biochemistry questions need different visual forms.
- Good scientific plots pair a visual pattern with a limitation.
- QC plots are as important as final result plots.

Common mistake:

- Do not let an attractive plot make the conclusion stronger than the data allow.

Limitations:

- All datasets are synthetic and for learning.
- These figures are not clinical, diagnostic, regulatory, or drug-efficacy evidence.
- Real projects need stronger metadata, validation, and domain review.
